# Cài đặt lại thuật toán FPClose bằng Julia

**Đồ án môn Khai thác dữ liệu và ứng dụng.** Nhóm cài đặt lại từ đầu thuật toán **FPClose**, khai thác tập phổ biến đóng (closed frequent itemsets) dựa trên cấu trúc FP-tree, theo mô tả trong bài báo gốc:

> G. Grahne và J. Zhu, "Fast Algorithms for Frequent Itemset Mining Using FP-Trees", *IEEE Transactions on Knowledge and Data Engineering*, vol. 17, no. 10, pp. 1347–1362, 2005. DOI: [10.1109/TKDE.2005.166](https://doi.org/10.1109/TKDE.2005.166)

Notebook này trình bày toàn bộ quy trình của đồ án: cài đặt môi trường, chuẩn bị dữ liệu, kiểm thử, đối chiếu kết quả với SPMF, thực nghiệm đánh giá và các ứng dụng thực tế.

**Thành viên nhóm:**
- Cao Hoàng Long — 24127442
- Nguyễn Khánh Linh — 24127197
- Hoàng Yến Trang — 24127133

**Tài nguyên:**
- Mã nguồn: https://github.com/DRAGON-1312/FPClose
- Bộ dữ liệu benchmark đã chuẩn hóa (Hugging Face): https://huggingface.co/datasets/hytrang/fpclose-fim-benchmarks

**Cách thực thi:** notebook được thiết kế để chạy trên Google Colab thông qua *Runtime → Run all*, với kernel **Python 3**; các bước sử dụng Julia được gọi qua lệnh shell `!julia ...`. Các bước cài đặt, tải mã nguồn và tải dữ liệu sẽ tự động được bỏ qua nếu đã thực hiện trước đó, nhờ vậy notebook có thể chạy lại nhiều lần. Mỗi mục tự kiểm tra các điều kiện cần thiết (dữ liệu, Java, thư viện Python) và chỉ phụ thuộc vào mục 1, do đó lỗi phát sinh ở một mục không ảnh hưởng đến các mục còn lại.

**Nội dung:**
1. Cài đặt môi trường
2. Chuẩn bị dữ liệu 
3. Kiểm thử tự động (unit test)
4. Minh họa FPClose qua dòng lệnh
5. Đối chiếu với SPMF
6. Thực nghiệm
7. Ứng dụng thực tế
8. Kết luận

## 1. Cài đặt môi trường

Notebook được thực thi trên **Google Colab** (hệ điều hành Ubuntu, sử dụng CPU, không yêu cầu GPU) với kernel **Python 3**. Môi trường thực thi gồm các thành phần sau:

| Thành phần | Phiên bản / nguồn | Mục đích sử dụng |
|---|---|---|
| Julia | **1.13.4**, bản phát hành chính thức (tải từ julialang.org, giải nén vào `/usr`) | Thực thi cài đặt FPClose của nhóm (`src/`, `demo.jl`, `scripts/`) |
| Gói Julia | Khai báo trong `Project.toml`, cài đặt bằng `Pkg.instantiate()` | Các phụ thuộc của mã nguồn và bộ kiểm thử |
| Java | **Temurin 25**, được `scripts/reference_check.jl` tự động tải vào `tools/` | Chỉ dùng để chạy SPMF phục vụ đối chiếu kết quả |
| SPMF | `spmf.jar`, được tự động tải vào `tools/` | Cài đặt tham chiếu để so sánh, không tham gia vào thuật toán của nhóm |
| Python | Python 3 có sẵn trên Colab, cùng `pandas` và `matplotlib` | Chỉ dùng để hiển thị bảng và hình |
| Dữ liệu | Hugging Face `hytrang/fpclose-fim-benchmarks` (mục 2) | 9 bộ benchmark, các tập con của Retail và dữ liệu tổng hợp |

Phiên bản Julia được chọn trùng với phiên bản trên máy dùng để đo kết quả chính thức (chi tiết tại `results/ENVIRONMENT.md`).

Ô cấu hình bên dưới khai báo địa chỉ repository (`REPO_URL`), nhánh mã nguồn (`BRANCH`) và phiên bản Julia (`JULIA_VERSION`). Bên cạnh đó, ô này định nghĩa một số hàm tiện ích được các mục sau sử dụng để tự chuẩn bị điều kiện thực thi:
- `sh(cmd)`: thực thi lệnh shell tại thư mục gốc của repository;
- `ensure_data([...])`: tải bổ sung các bộ dữ liệu còn thiếu;
- `ensure_java()`: chuẩn bị Java 25 và `spmf.jar` phục vụ việc chạy SPMF;
- `need_py(...)`: cài đặt các thư viện Python còn thiếu.

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL = 'https://github.com/DRAGON-1312/FPClose.git'
BRANCH = 'feature/colab-demo'            # đổi thành nhánh khác (ví dụ 'feature/fast-reader') để chạy thử trước khi merge
JULIA_VERSION = '1.13.4'   # cùng phiên bản với máy đo chính thức (results/ENVIRONMENT.md)
REPO_DIR = '/content/FPClose'
MACHINE_LABEL = 'colab'    # nhãn cột machine cho các số đo minh họa trên Colab

import os, glob, time, importlib
os.environ['FPCLOSE_MACHINE'] = MACHINE_LABEL
T_START = time.time()

# ---- Hàm tiện ích: mỗi mục sau tự kiểm tra/chuẩn bị điều kiện của mình bằng các hàm này ----
def sh(cmd):
    """Chạy lệnh shell (từ gốc repo nếu đã clone), output hiện ngay trong ô."""
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
    get_ipython().system(cmd)

def ensure_data(names):
    """Bảo đảm có data/benchmark/<name>.txt; thiếu thì gọi scripts/data_setup.jl --only <các file thiếu>."""
    path = lambda n: os.path.join(REPO_DIR, 'data', 'benchmark', f'{n}.txt')
    missing = [n for n in names if not os.path.exists(path(n))]
    if missing:
        print('Thiếu dữ liệu', missing, '-> chạy scripts/data_setup.jl --only', ','.join(missing))
        sh('julia --project=. scripts/data_setup.jl --only ' + ','.join(missing))
    ok = all(os.path.exists(path(n)) for n in names)
    print('Dữ liệu', names, 'sẵn sàng' if ok else 'VẪN THIẾU, bỏ qua bước cần dữ liệu này')
    return ok

def ensure_java():
    """Bảo đảm có Java 25 và tools/spmf.jar bằng setup_tools() có sẵn trong scripts/reference_check.jl
    (trên Linux tự tải Temurin 25), rồi thêm JDK vào PATH: compare_spmf.jl và run_benchmark_grid.jl gọi `java` trên PATH."""
    jdk_bins = lambda: sorted(glob.glob(os.path.join(REPO_DIR, 'tools', 'jdk-25*', 'bin')))
    if not jdk_bins() or not os.path.exists(os.path.join(REPO_DIR, 'tools', 'spmf.jar')):
        sh('''julia --project=. -e 'include("scripts/reference_check.jl"); setup_tools(TOOLS_DIR; with_fimi = false)' ''')
    bins = jdk_bins()
    if bins and bins[-1] not in os.environ['PATH'].split(os.pathsep):
        os.environ['PATH'] = bins[-1] + os.pathsep + os.environ['PATH']
    print('JDK:', bins[-1] if bins else 'không thấy tools/jdk-25*, dùng java có sẵn trên máy')
    sh('java -version 2>&1 | head -n 1')

def need_py(*pkgs):
    """Import được thì thôi, không thì pip install (Colab đã có sẵn pandas, IPython)."""
    for p in pkgs:
        try:
            importlib.import_module(p)
        except ImportError:
            sh(f'pip install -q {p}')

print('Nhánh:', BRANCH, '| Julia', JULIA_VERSION, '| thư mục', REPO_DIR)

In [ ]:
# Cài Julia từ tarball chính thức vào /usr (bỏ qua nếu `julia --version` đã đúng phiên bản).
import subprocess
def julia_version(exe='julia'):
    try:
        return subprocess.run([exe, '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return ''

WANT = f'julia version {JULIA_VERSION}'
if julia_version() == WANT:
    print('Đã có', julia_version(), '-> bỏ qua bước cài')
else:
    if julia_version('/usr/bin/julia') != WANT:
        major_minor = '.'.join(JULIA_VERSION.split('.')[:2])
        url = f'https://julialang-s3.julialang.org/bin/linux/x64/{major_minor}/julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
        print('Tải', url)
        sh(f'curl -fsSL -o /tmp/julia.tar.gz {url} && tar -xzf /tmp/julia.tar.gz -C /usr --strip-components 1 && rm -f /tmp/julia.tar.gz')
    # Colab có sẵn một bản Julia khác ở /usr/local/bin (đứng trước /usr/bin trong PATH): trỏ lệnh `julia` về bản
    # vừa cài, CHỈ khi /usr/bin/julia đúng phiên bản (tải lỗi thì giữ nguyên bản có sẵn).
    if julia_version('/usr/bin/julia') == WANT and os.path.isdir('/usr/local/bin'):
        sh('ln -sf /usr/bin/julia /usr/local/bin/julia')
    if julia_version() != WANT:
        print('CẢNH BÁO: lệnh julia là', julia_version() or '(không tìm thấy)', 'thay vì', JULIA_VERSION)
!julia --version

In [ ]:
# Clone repo (bỏ qua nếu đã có) rồi chuyển sang nhánh BRANCH. Từ đây mọi lệnh chạy từ gốc repo.
![ -d {REPO_DIR}/.git ] || git clone --quiet {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin {BRANCH} && git -C {REPO_DIR} checkout --quiet -B {BRANCH} FETCH_HEAD
%cd {REPO_DIR}
!git log -1 --format='%h %s (%ci)'

In [ ]:
%%time
# Cài gói theo Project.toml (repo chỉ dùng thư viện chuẩn của Julia), biên dịch trước, in versioninfo.
!julia --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.precompile(); using FPClose, InteractiveUtils; versioninfo()'

## 2. Chuẩn bị dữ liệu

Toàn bộ quá trình chuẩn bị dữ liệu được nhóm cài đặt trong `scripts/data_setup.jl`, gồm ba bước:
1. Tải 9 bộ dữ liệu benchmark đã chuẩn hóa (Chess, Mushrooms, Retail, T10I4D100K, Accidents, Connect, Kosarak, Pumsb*, T40I10D100K) từ Hugging Face `hytrang/fpclose-fim-benchmarks` và kiểm tra mã băm SHA-256 nhằm bảo đảm tính toàn vẹn của dữ liệu.
2. Sinh 5 tập con của Retail (10%, 25%, 50%, 75% và 100% số giao dịch) tại `data/scalability/`, phục vụ thực nghiệm khả năng mở rộng (scalability).
3. Sinh 6 cơ sở dữ liệu (CSDL) tổng hợp có độ dài giao dịch trung bình tăng dần từ 5 đến 30 tại `data/synthetic/`, phục vụ thực nghiệm về ảnh hưởng của độ dài giao dịch.

Các file sinh ra ở bước 2 và 3 cũng được kiểm tra mã băm để bảo đảm khả năng tái lập. Mọi tham số (nguồn tải, mã băm, seed sinh dữ liệu) được lưu trong `data/meta/*.csv`. Trường hợp dữ liệu đã đầy đủ, bước này sẽ được bỏ qua. Các mục sau sử dụng dữ liệu được chuẩn bị tại đây; nếu phát hiện thiếu dữ liệu, mỗi mục sẽ tự tải bổ sung thông qua `data_setup.jl --only ...`. Hai bảng bên dưới trình bày thống kê của các bộ dữ liệu thu được.

In [ ]:
%%time
count_files = lambda pattern: len(glob.glob(os.path.join(REPO_DIR, pattern)))
n_bench = count_files('data/benchmark/*.txt')
n_scal = count_files('data/scalability/*.txt')
n_syn = count_files('data/synthetic/*.txt')
print(f'Hiện có: {n_bench}/9 benchmark, {n_scal}/5 tập con Retail, {n_syn}/6 synthetic')
if n_bench >= 9 and n_scal >= 5 and n_syn >= 6:
    print('Đủ dữ liệu -> bỏ qua data_setup.jl')
else:
    sh('julia --project=. scripts/data_setup.jl')

In [ ]:
need_py('pandas')
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
for p in ['data/meta/dataset_statistics.csv', 'data/meta/synthetic_statistics.csv']:
    full = os.path.join(REPO_DIR, p)
    print('==', p)
    display(pd.read_csv(full)) if os.path.exists(full) else print('  (không có file)')

**Kết quả.** Dữ liệu được chuẩn bị đầy đủ (9 bộ benchmark, 5 tập con của Retail, 6 CSDL tổng hợp) và đều khớp mã băm. Các bộ dữ liệu gồm cả loại dày đặc và loại thưa, phục vụ việc đánh giá thuật toán trên nhiều đặc điểm dữ liệu khác nhau.

## 3. Kiểm thử tự động (unit test)

Mục này thực thi toàn bộ bộ kiểm thử `test/runtests.jl`, bao gồm:
- kiểm tra bộ đọc file định dạng SPMF và các tham số dòng lệnh;
- đối chiếu kết quả với đáp án vét cạn (brute force) trên các CSDL sinh ngẫu nhiên;
- xác nhận mọi tổ hợp bật/tắt các tối ưu đều cho cùng một kết quả;
- so sánh từng itemset và support với output của SPMF đã được lưu sẵn trong `test/fixtures/spmf/` (không cần chạy Java);
- kiểm tra bước khôi phục mọi tập phổ biến từ tập đóng (recovery) và bước sinh luật kết hợp;
- chạy `fpclose` ở minsup (ngưỡng support tối thiểu) 90% trên 9 bộ benchmark đã tải ở mục 2; nhóm test này được bỏ qua nếu mục 2 không tải được dữ liệu.

Dữ liệu ngẫu nhiên trong kiểm thử sử dụng seed cố định nhằm bảo đảm khả năng tái lập. Khi toàn bộ test đều đạt, Julia chỉ hiển thị bảng **Test Summary** dạng thu gọn, gồm số test đạt trên tổng số test; trường hợp có test không đạt, chi tiết lỗi cùng bảng kết quả của từng nhóm test sẽ được hiển thị đầy đủ.

In [ ]:
%%time
!julia --project=. test/runtests.jl 2>&1

**Kết quả.** Toàn bộ 28.266 test đều đạt, không có test lỗi.

## 4. Minh họa FPClose qua dòng lệnh

**Ví dụ nhỏ.** CSDL `data/toy/pasquier99.txt` gồm 5 giao dịch, là ví dụ minh họa được sử dụng trong tài liệu của SPMF. Với minsup bằng 40%, chương trình xuất ra ba file:
- các tập phổ biến đóng (closed frequent itemsets);
- mọi tập phổ biến, được khôi phục từ các tập đóng (bước recovery);
- 10 luật kết hợp (association rules) có lift cao nhất trong số các luật có confidence ≥ 0,6.

Mỗi dòng output có dạng `1 2 3 #SUP: 4`, biểu thị itemset {1, 2, 3} xuất hiện trong 4 giao dịch. Thời gian thực thi của từng giai đoạn được in kèm theo.

**Dữ liệu thực tế.** Tiếp theo, chương trình được chạy theo đúng dạng lệnh `demo.jl --minsup <tỉ lệ> data/benchmark/<file>.txt` trên bộ Chess với minsup bằng 0,8.

In [ ]:
!mkdir -p out
!echo '== Input: data/toy/pasquier99.txt' && cat data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 40% --output-closed out/demo_closed.txt --output-all out/demo_all.txt --rules out/demo_rules.txt --minconf 0.6 --top 10 data/toy/pasquier99.txt

In [ ]:
import os
for title, path in [('Tập phổ biến đóng', 'out/demo_closed.txt'),
                    ('Mọi tập phổ biến', 'out/demo_all.txt'),
                    ('Luật kết hợp top-10 theo lift', 'out/demo_rules.txt')]:
    print('=' * 10, title, f'({path})')
    print(open(path, encoding='utf-8').read() if os.path.exists(path) else '(không có file)')

In [ ]:
%%time
# Chạy theo đúng dạng lệnh của đề: julia --project=. demo.jl --minsup <tỉ lệ> data/benchmark/<file>.txt
# Tập đóng được in ra stdout và chuyển vào file; thời gian từng giai đoạn hiển thị trong ô (stderr).
if ensure_data(['chess']):
    os.makedirs(os.path.join(REPO_DIR, 'out'), exist_ok=True)
    sh('julia --project=. demo.jl --minsup 0.8 data/benchmark/chess.txt > out/chess_0.8_closed.txt')
    out_path = os.path.join(REPO_DIR, 'out', 'chess_0.8_closed.txt')
    if os.path.exists(out_path):
        lines = [l for l in open(out_path, encoding='utf-8').read().splitlines() if '#SUP:' in l]
        print('Số tập phổ biến đóng (Chess, minsup 0,8):', len(lines))
        print('5 dòng đầu:')
        print('\n'.join(lines[:5]))

**Kết quả.** Trên `pasquier99.txt` (minsup 40%), chương trình tìm được 5 tập phổ biến đóng, trùng với ví dụ trong tài liệu SPMF. Trên Chess (minsup 0,8), chương trình tìm được 5083 tập phổ biến đóng; kết quả này được đối chiếu với SPMF tại mục 5.

## 5. Đối chiếu với SPMF

SPMF là thư viện Java có sẵn cài đặt FPClose và được nhóm sử dụng làm **cài đặt tham chiếu** để đánh giá tính đúng đắn. Mục này sử dụng hai bộ dữ liệu Chess và Mushrooms đã chuẩn bị ở mục 2 (nếu thiếu, dữ liệu sẽ được tải bổ sung tự động). Quá trình đối chiếu gồm hai bước:

1. **Kiểm tra công cụ tham chiếu** (`scripts/reference_check.jl`): Java 25 (Temurin) và `spmf.jar` được tải về thư mục `tools/`; SPMF được chạy trên Chess và Mushrooms, sau đó số tập đóng của SPMF được so với số liệu tham chiếu. Kết quả được ghi vào file riêng `results/reference/spmf_check_chess+mushrooms.csv`, không ghi đè kết quả đã lưu trong repository. Bước biên dịch cài đặt tham khảo FIMI được bỏ qua (tùy chọn `--skip-fimi`) nhằm rút ngắn thời gian thực thi.
2. **So sánh cài đặt của nhóm với SPMF** (`scripts/compare_spmf.jl`): từng itemset cùng support tương ứng được so sánh giữa hai cài đặt, trên toàn bộ CSDL nhỏ trong `data/toy/` và trên Chess ở minsup 80%. Hai cài đặt được xem là khớp hoàn toàn khi không tồn tại itemset chỉ xuất hiện ở một bên và không có itemset nào lệch support.

Giữa hai bước, hàm `ensure_java()` bổ sung Java 25 vào biến môi trường `PATH`, do `compare_spmf.jl` gọi trực tiếp lệnh `java` của hệ thống.

In [ ]:
ensure_data(['chess', 'mushrooms'])

In [ ]:
%%time
!julia --project=. scripts/reference_check.jl --datasets chess,mushrooms --skip-fimi

In [ ]:
need_py('pandas')
import os, pandas as pd
p = 'results/reference/spmf_check_chess+mushrooms.csv'
display(pd.read_csv(p)) if os.path.exists(p) else print('Chưa có', p)

In [ ]:
%%time
# Bảng khớp từng itemset trên mọi CSDL toy (closed và mọi tập phổ biến), sau đó một cấu hình của chess (80%)
ensure_java()
!julia --project=. scripts/compare_spmf.jl --table
!julia --project=. scripts/compare_spmf.jl data/benchmark/chess.txt 80%

**Kết quả.** Cài đặt của nhóm khớp hoàn toàn với SPMF về từng itemset và support trên mọi CSDL nhỏ trong `data/toy/` và trên Chess ở minsup 80%.

## 6. Thực nghiệm

### 6a. Tái hiện một phần thực nghiệm trên Colab

Nhằm minh họa quy trình đo, `scripts/run_benchmark_grid.jl` được chạy ở chế độ rút gọn: chỉ trên bộ Chess, với 2–3 mức minsup và mỗi cấu hình đo một lần, gồm phần kiểm tra tính đúng đắn và phần đo thời gian so với SPMF. Các file kết quả mang hậu tố `_quick`, vì vậy không ghi đè kết quả chính thức.

Số đo trên Colab chỉ mang tính minh họa do phụ thuộc vào cấu hình máy ảo. Số liệu chính thức của báo cáo là kết quả đo đầy đủ trên máy MSI, được trình bày tại mục 6b.

In [ ]:
%%time
if ensure_data(['chess']):
    ensure_java()
    sh(f'julia --project=. scripts/run_benchmark_grid.jl --quick --datasets chess --only correctness,runtime --machine {MACHINE_LABEL}')

In [ ]:
need_py('pandas')
import os, pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
for p in ['results/correctness/grid_correctness_quick.csv',
          'results/runtime/grid_runtime_summary_quick.csv']:
    print('==', p)
    display(pd.read_csv(p)) if os.path.exists(p) else print('  (chưa có file)')

**Kết quả.** Quy trình đo hoạt động đầy đủ trên Colab và mọi cấu hình đều khớp với SPMF. Thời gian tuyệt đối khác với số đo chính thức do khác biệt phần cứng.

### 6b. Kết quả đo chính thức (máy MSI)

Toàn bộ số liệu Chương 2 trong báo cáo được đo trên máy MSI Vector A16; cấu hình chi tiết được ghi tại `results/ENVIRONMENT.md`. Quy ước đo như sau:
- thời gian là trung vị của 3 lần chạy, với giới hạn 600 giây cho mỗi cấu hình;
- khi so sánh với SPMF, thời gian của cả hai cài đặt đều bao gồm bước đọc dữ liệu và ghi file kết quả;
- bộ nhớ của nhóm là mức tăng bộ nhớ đỉnh (peak RSS) so với mức nền của Julia; bộ nhớ của SPMF là heap của JVM, do đó không được so sánh trực tiếp với số liệu của nhóm.

Các bảng tổng hợp được sinh bởi `scripts/summarize_results.jl` và các hình được vẽ bởi `scripts/plot_results.jl`. Phần phân tích chi tiết được trình bày tại Chương 2 của báo cáo (`docs/Report.pdf`).

In [ ]:
print(open('results/ENVIRONMENT.md', encoding='utf-8').read())

In [ ]:
need_py('pandas')
import os, pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
SUMMARY = 'results/summary'

def show(name, title, cols=None):
    path = f'{SUMMARY}/{name}.csv'
    if not os.path.exists(path):
        print('Chưa có', path)
        return
    df = pd.read_csv(path)
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    print(f'== {title} ({path}, {len(df)} dòng)')
    display(df)

show('correctness_summary', 'Tính đúng đắn: cài đặt của nhóm so với SPMF')
show('speedup', 'Thời gian: cài đặt của nhóm so với SPMF (tỷ số SPMF / nhóm > 1 nghĩa là nhóm nhanh hơn)')
show('time_breakdown', 'Phân rã thời gian của cài đặt nhóm')
show('ablation', 'Ablation các tối ưu (% so với optimized)',
     cols=['dataset', 'minsup_pct', 'variant', 'n_closed', 'mine_s', 'mine_s_vs_opt_pct',
           'mem_delta_mb', 'mem_delta_mb_vs_opt_pct', 'alloc_mb', 'alloc_mb_vs_opt_pct'])
show('scalability', 'Khả năng mở rộng (Retail 10%–100%)')
show('txlen', 'Ảnh hưởng độ dài giao dịch')

In [ ]:
import os
from IPython.display import Image, Markdown, display
FIG = 'results/figures'
figures = [
    ('runtime_grid', 'Thời gian chạy theo minsup trên 9 bộ dữ liệu (nhóm và SPMF)', 900),
    ('itemsets_vs_minsup', 'Số tập phổ biến đóng theo minsup', 900),
    ('time_breakdown', 'Phân rã thời gian của cài đặt nhóm', 600),
    ('memory_variants_grid', 'Bộ nhớ thực dùng theo biến thể', 900),
    ('time_variants_grid', 'Thời gian khai thác theo biến thể', 900),
    ('ablation_time_relative', 'Ablation thời gian (so với optimized)', 600),
    ('scalability_time', 'Khả năng mở rộng theo kích thước', 600),
    ('txlen_time', 'Thời gian theo độ dài giao dịch', 600),
    ('txlen_itemsets', 'Số tập đóng theo độ dài giao dịch', 600),
]
for name, caption, width in figures:
    path = f'{FIG}/{name}.png'
    if os.path.exists(path):
        display(Markdown(f'**{caption}** (`{path}`)'))
        display(Image(filename=path, width=width))
    else:
        print('Chưa có', path)

**Nhận xét.** Cài đặt của nhóm khớp 100% với SPMF trên toàn bộ 56 cấu hình thuộc 9 bộ benchmark. Về thời gian, cài đặt của nhóm nhanh hơn SPMF ở 47/56 cấu hình (trung bình nhân khoảng 1,72 lần), với ưu thế rõ nhất trên dữ liệu dày đặc. Phân tích chi tiết được trình bày tại Chương 2 của báo cáo.

## 7. Ứng dụng thực tế (Chương 3)

Hai ứng dụng được nhóm thực hiện trực tiếp trong notebook này. Toàn bộ itemset và luật kết hợp đều được sinh bằng chính cài đặt FPClose của nhóm, thông qua dòng lệnh `demo.jl` hoặc các hàm trong `src/` (`read_spmf`, `fpclose`, `recover_all_frequent`, `generate_rules`, ...); nhóm không sử dụng mlxtend hay bất kỳ thư viện khai thác tập phổ biến nào khác. Các thư viện xử lý ảnh, phân cụm k-means và pandas chỉ được dùng ở bước biến đổi dữ liệu thành giao dịch và trình bày kết quả.

### 7.1 Ứng dụng 1 – Market basket

*Nội dung được bổ sung cùng Chương 3.*

- **Mục tiêu:** tìm các nhóm sản phẩm thường được mua cùng nhau và luật kết hợp (support, confidence, lift) để gợi ý bán chéo / sắp xếp kệ hàng.
- **Dữ liệu:** (sẽ bổ sung) tên bộ dữ liệu giỏ hàng, nguồn, giấy phép, kích thước; cách tải nằm trong notebook.
- **Biến đổi dữ liệu thành giao dịch (đặt trong notebook):** gom các dòng theo mã hóa đơn thành tập mã sản phẩm; bỏ hóa đơn hủy, số lượng âm, mã không phải sản phẩm; ánh xạ mỗi sản phẩm sang một số nguyên ≥ 0 và lưu bảng ánh xạ để dịch ngược; ghi file định dạng SPMF (mỗi dòng một giao dịch, item cách nhau dấu cách).
- **Các bước:**
  1. Thống kê dữ liệu giao dịch (số giao dịch, số item, độ dài trung bình).
  2. Chọn minsup và minconf, có giải thích.
  3. Chạy FPClose của nhóm: `demo.jl --minsup ... --output-closed ... --rules ... --minconf ... --top k`.
  4. Dịch item về tên sản phẩm; trình bày các tập đóng tiêu biểu và các luật có lift cao nhất.
  5. Nhận xét ý nghĩa kinh doanh và hạn chế.

### 7.2 Ứng dụng 2 – Visual patterns (BoVW)

*Nội dung được bổ sung cùng Chương 3.*

- **Mục tiêu:** tìm các tổ hợp "visual word" thường cùng xuất hiện trong ảnh của từng lớp, tức các mẫu thị giác đặc trưng của lớp đó (Bag of Visual Words).
- **Dữ liệu:** (sẽ bổ sung) bộ ảnh nhỏ có nhãn lớp (tên, nguồn, giấy phép, số ảnh dùng); cách tải nằm trong notebook.
- **Biến đổi dữ liệu thành giao dịch (đặt trong notebook):** trích đặc trưng cục bộ của từng ảnh (ví dụ ORB/SIFT bằng OpenCV); gom cụm toàn bộ descriptor bằng k-means (ví dụ scikit-learn) để lập từ điển k visual word; mỗi ảnh thành một giao dịch là tập id các visual word xuất hiện trong ảnh (có thể lọc word quá phổ biến); ghi file định dạng SPMF, mỗi lớp một file. OpenCV và k-means chỉ dùng ở bước này.
- **Các bước:**
  1. Chọn k và cách lọc word, có giải thích; thống kê giao dịch thu được.
  2. Chạy FPClose của nhóm trên giao dịch của từng lớp.
  3. So sánh các tập đóng giữa các lớp, tìm tập đặc trưng riêng của mỗi lớp.
  4. Trực quan hóa: vẽ lại vị trí các keypoint thuộc một tập đóng lên vài ảnh mẫu.
  5. Nhận xét và hạn chế.

## 8. Kết luận

(Sẽ bổ sung) tóm tắt kết quả (tính đúng đắn so với SPMF, so sánh thời gian, ảnh hưởng của các tối ưu) và hướng phát triển.

In [ ]:
print(f'Tổng thời gian chạy notebook: {(time.time() - T_START) / 60:.1f} phút')